
# 🌍 Lab 4: การประมวลผลข้อมูลด้วย NumPy และ GeoPandas
## วิชา GE 234 Basic Programming for Geographers

### 🎯 **วัตถุประสงค์**
1. เข้าใจการใช้ **NumPy** สำหรับการประมวลผลข้อมูลทางภูมิศาสตร์ เช่น ข้อมูล Raster และพิกัด
2. ใช้ **GeoPandas** ในการจัดการและวิเคราะห์ข้อมูลเวกเตอร์ เช่น **Shapefile**
3. สามารถดำเนินการทางสถิติกับข้อมูลพิกัดและชั้นข้อมูลทางภูมิศาสตร์ได้
4. สามารถใช้ NumPy และ GeoPandas ร่วมกันเพื่อวิเคราะห์ข้อมูลได้

---

## 🔹 ตัวอย่างที่ 1: ใช้ NumPy คำนวณข้อมูล NDVI จากภาพ Raster


In [59]:

import numpy as np

# สร้างข้อมูลตัวอย่างสำหรับภาพ NDVI (Normalized Difference Vegetation Index)
nir = np.array([[0.7, 0.8, 0.6], [0.9, 0.5, 0.3], [0.4, 0.7, 0.2]])
red = np.array([[0.3, 0.4, 0.2], [0.5, 0.3, 0.1], [0.2, 0.3, 0.1]])

# คำนวณค่า NDVI
ndvi = (nir - red) / (nir + red)

print("ค่า NDVI:")
print(ndvi)


ค่า NDVI:
[[0.4        0.33333333 0.5       ]
 [0.28571429 0.25       0.5       ]
 [0.33333333 0.4        0.33333333]]



## 🔹 ตัวอย่างที่ 2: ใช้ NumPy คำนวณค่าเฉลี่ยและค่ามากสุดของ NDVI


In [60]:

print(f"ค่าเฉลี่ย NDVI: {np.mean(ndvi):.2f}")
print(f"ค่า NDVI สูงสุด: {np.max(ndvi):.2f}")
print(f"ค่า NDVI ต่ำสุด: {np.min(ndvi):.2f}")


ค่าเฉลี่ย NDVI: 0.37
ค่า NDVI สูงสุด: 0.50
ค่า NDVI ต่ำสุด: 0.25



## 🔹 ตัวอย่างที่ 3: ใช้ GeoPandas โหลดและวิเคราะห์ข้อมูล Shapefile


In [61]:

import geopandas as gpd

# โหลดข้อมูลชั้นข้อมูลจังหวัดของประเทศไทย
gdf = gpd.read_file("https://data.opendevelopmentmekong.net/th/dataset/8f3fa1b8-cb5c-48c8-9fd7-d3c213ea23db/resource/1559cee4-fedc-4330-be9c-d8cf3dd75015/download/tha_admbnda_adm1_rtsd_20190221.zip")

# แสดงข้อมูล 5 แถวแรก
print(gdf.head())


   Shape_Leng  Shape_Area        ADM1_EN        ADM1_TH ADM1_PCODE ADM1_REF  \
0    3.927244    0.275313  Amnat Charoen     อำนาจเจริญ       TH37     None   
1    1.739908    0.079210      Ang Thong        อ่างทอง       TH15     None   
2    2.417227    0.131339        Bangkok  กรุงเทพมหานคร       TH10     None   
3    4.414998    0.340784      Bueng Kan         บึงกาฬ       TH38     None   
4    8.701860    0.844537       Buri Ram      บุรีรัมย์       TH31     None   

  ADM1ALT1EN ADM1ALT2EN ADM1ALT1TH ADM1ALT2TH   ADM0_EN    ADM0_TH ADM0_PCODE  \
0       None       None       None       None  Thailand  ประเทศไทย         TH   
1       None       None       None       None  Thailand  ประเทศไทย         TH   
2       None       None       None       None  Thailand  ประเทศไทย         TH   
3       None       None       None       None  Thailand  ประเทศไทย         TH   
4       None       None       None       None  Thailand  ประเทศไทย         TH   

        date    validOn validTo  \
0 2


## 🔹 ตัวอย่างที่ 4: คำนวณพื้นที่ของแต่ละจังหวัด


In [62]:
# ตรวจสอบค่า CRS (Coordinate Reference System)
print(gdf.crs)

# แปลง CRS ไปยังระบบพิกัดเชิงเส้น (Projected CRS) สำหรับการคำนวณพื้นที่ที่ถูกต้อง
# EPSG:32647 คือ UTM Zone 47N ซึ่งเหมาะสมกับพื้นที่ในประเทศไทย
gdf_proj = gdf.to_crs(epsg=32647)

# คำนวณพื้นที่ของแต่ละจังหวัด (หน่วยเป็นตารางเมตร) และแปลงเป็นตารางกิโลเมตร
gdf_proj["area_sqkm"] = gdf_proj.geometry.area / 1e6

# แสดงพื้นที่จังหวัด 5 อันดับแรกที่ใหญ่ที่สุด
# ใช้คอลัมน์ ADM1_TH แทน PROV_NAME
print(gdf_proj.nlargest(5, "area_sqkm")[["ADM1_TH", "area_sqkm"]])

EPSG:4326
        ADM1_TH     area_sqkm
9     เชียงใหม่  22159.519971
28   นครราชสีมา  20750.869676
15    กาญจนบุรี  19436.356316
68          ตาก  17266.546740
71  อุบลราชธานี  15636.863442



## 🔹 ตัวอย่างที่ 5: ใช้ GeoPandas ทำ Spatial Join


In [63]:

# โหลดข้อมูลชั้นข้อมูลอำเภอ
gdf_districts = gpd.read_file("https://data.opendevelopmentmekong.net/th/dataset/8f3fa1b8-cb5c-48c8-9fd7-d3c213ea23db/resource/1559cee4-fedc-4330-be9c-d8cf3dd75015/download/tha_admbnda_adm1_rtsd_20190221.zip")
# ทำ Spatial Join ระหว่างอำเภอกับจังหวัด
gdf_joined = gpd.sjoin(gdf_districts, gdf, how="inner", predicate="within")

# แสดงตัวอย่างข้อมูลที่เชื่อมโยงกัน
print(gdf_joined.head())


   Shape_Leng_left  Shape_Area_left   ADM1_EN_left   ADM1_TH_left  \
0         3.927244         0.275313  Amnat Charoen     อำนาจเจริญ   
1         1.739908         0.079210      Ang Thong        อ่างทอง   
2         2.417227         0.131339        Bangkok  กรุงเทพมหานคร   
3         4.414998         0.340784      Bueng Kan         บึงกาฬ   
4         8.701860         0.844537       Buri Ram      บุรีรัมย์   

  ADM1_PCODE_left ADM1_REF_left ADM1ALT1EN_left ADM1ALT2EN_left  \
0            TH37          None            None            None   
1            TH15          None            None            None   
2            TH10          None            None            None   
3            TH38          None            None            None   
4            TH31          None            None            None   

  ADM1ALT1TH_left ADM1ALT2TH_left  ... ADM1ALT1EN_right ADM1ALT2EN_right  \
0            None            None  ...             None             None   
1            None            N


# 📝 **กิจกรรมในแลป**

1. **แบบฝึกหัด 1**: ใช้ NumPy คำนวณค่า **Mean, Max, Min** ของค่า NDVI ในอาร์เรย์ที่สร้างขึ้นเอง
2. **แบบฝึกหัด 2**: ใช้ GeoPandas โหลด **Shapefile** ของจังหวัด และคำนวณพื้นที่ของแต่ละจังหวัด
3. **แบบฝึกหัด 3**: ใช้ GeoPandas ทำ **Spatial Join** ระหว่างข้อมูลจังหวัดและอำเภอ
4. **แบบฝึกหัด 4**: ใช้ NumPy และ GeoPandas ร่วมกันเพื่อหาข้อมูลจังหวัดที่มี NDVI เฉลี่ยสูงสุด


### แบบฝึกหัดที่ 1

In [64]:
import numpy as np

# สร้างข้อมูลตัวอย่างสำหรับภาพ NDVI (Normalized Difference Vegetation Index)
nir = np.array([[0.5, 0.8, 0.6, 0.1 ,0.2], [0.9, 0.5, 0.3, 0.8, 0.9], [0.4, 0.7, 0.2, 0.3, 0.4]])
red = np.array([[0.3, 0.4, 0.2, 0.6 , 0.7], [0.5, 0.3, 0.1, 0.5, 0.6], [0.2, 0.3, 0.1, 0.1, 0.2]])

# คำนวณค่า NDVI
ndvi = (nir - red) / (nir + red)

In [65]:
import numpy as np

# ตรวจสอบให้แน่ใจว่า ndvi ถูกคำนวณจากเซลล์ก่อนหน้านี้
# ถ้า (nir + red) มีค่าเป็นศูนย์ ควรจัดการเพื่อป้องกัน ZeroDivisionError
# ในกรณีนี้, numpy จะจัดการค่า NaN หรือ Inf โดยอัตโนมัติหากมีการหารด้วยศูนย์
# แต่เพื่อความปลอดภัย เราสามารถใช้ np.errstate หรือกรองค่าก่อนได้

# คำนวณค่าเฉลี่ย ค่าสูงสุด และค่าต่ำสุดของ ndvi
mean_ndvi = np.mean(ndvi)
max_ndvi = np.max(ndvi)
min_ndvi = np.min(ndvi)

print(f"ค่าเฉลี่ย NDVI: {mean_ndvi:.2f}")
print(f"ค่า NDVI สูงสุด: {max_ndvi:.2f}")
print(f"ค่า NDVI ต่ำสุด: {min_ndvi:.2f}")

ค่าเฉลี่ย NDVI: 0.21
ค่า NDVI สูงสุด: 0.50
ค่า NDVI ต่ำสุด: -0.71


### แบบฝึกหัดที่ 2

In [66]:
import geopandas as gpd

# โหลดข้อมูลชั้นข้อมูลจังหวัดของประเทศไทย
# ใช้ URL เดียวกันกับตัวอย่างที่ 3
gdf_exercise2 = gpd.read_file("https://data.opendevelopmentmekong.net/th/dataset/8f3fa1b8-cb5c-48c8-9fd7-d3c213ea23db/resource/1559cee4-fedc-4330-be9c-d8cf3dd75015/download/tha_admbnda_adm1_rtsd_20190221.zip")

print("ข้อมูลจังหวัด 5 แถวแรก:")
print(gdf_exercise2.head())

# ตรวจสอบค่า CRS (Coordinate Reference System) ของข้อมูลที่โหลดมา
print(f"\nCRS ปัจจุบัน: {gdf_exercise2.crs}")

# แปลง CRS ไปยังระบบพิกัดเชิงเส้น (Projected CRS) สำหรับการคำนวณพื้นที่ที่ถูกต้อง
# EPSG:32647 คือ UTM Zone 47N ซึ่งเหมาะสมกับพื้นที่ในประเทศไทย
gdf_proj_exercise2 = gdf_exercise2.to_crs(epsg=32647)

# คำนวณพื้นที่ของแต่ละจังหวัด (หน่วยเป็นตารางเมตร) และแปลงเป็นตารางกิโลเมตร
gdf_proj_exercise2["area_sqkm"] = gdf_proj_exercise2.geometry.area / 1e6

# แสดงข้อมูลจังหวัดพร้อมพื้นที่ 5 อันดับแรกที่ใหญ่ที่สุด
print("\n5 จังหวัดที่มีพื้นที่มากที่สุด (ตารางกิโลเมตร):")
print(gdf_proj_exercise2.nlargest(5, "area_sqkm")[["ADM1_TH", "area_sqkm"]])

ข้อมูลจังหวัด 5 แถวแรก:
   Shape_Leng  Shape_Area        ADM1_EN        ADM1_TH ADM1_PCODE ADM1_REF  \
0    3.927244    0.275313  Amnat Charoen     อำนาจเจริญ       TH37     None   
1    1.739908    0.079210      Ang Thong        อ่างทอง       TH15     None   
2    2.417227    0.131339        Bangkok  กรุงเทพมหานคร       TH10     None   
3    4.414998    0.340784      Bueng Kan         บึงกาฬ       TH38     None   
4    8.701860    0.844537       Buri Ram      บุรีรัมย์       TH31     None   

  ADM1ALT1EN ADM1ALT2EN ADM1ALT1TH ADM1ALT2TH   ADM0_EN    ADM0_TH ADM0_PCODE  \
0       None       None       None       None  Thailand  ประเทศไทย         TH   
1       None       None       None       None  Thailand  ประเทศไทย         TH   
2       None       None       None       None  Thailand  ประเทศไทย         TH   
3       None       None       None       None  Thailand  ประเทศไทย         TH   
4       None       None       None       None  Thailand  ประเทศไทย         TH   

        date  

### แบบฝึกหัดที่ 3:

In [67]:
import geopandas as gpd

# โหลดข้อมูลชั้นข้อมูลอำเภอ (ADM2) ของประเทศไทย
# ใช้ URL จาก GADM ที่รวมทุกระดับชั้นข้อมูล แล้ว GeoPandas จะเลือกชั้นข้อมูลที่เหมาะสมให้
gdf_districts_exercise3 = gpd.read_file("https://geodata.ucdavis.edu/gadm/gadm4.1/shp/gadm41_THA_shp.zip", layer="gadm41_THA_2")

print("ข้อมูลอำเภอ 5 แถวแรก:")
print(gdf_districts_exercise3.head())

# ตรวจสอบ CRS ของข้อมูลอำเภอ
print(f"\nCRS ของข้อมูลอำเภอ: {gdf_districts_exercise3.crs}")

ข้อมูลอำเภอ 5 แถวแรก:
       GID_2 GID_0   COUNTRY    GID_1         NAME_1          NL_NAME_1  \
0  THA.1.1_1   THA  Thailand  THA.1_1  Amnat Charoen  จังหวัดอำนาจเจริญ   
1  THA.1.2_1   THA  Thailand  THA.1_1  Amnat Charoen  จังหวัดอำนาจเจริญ   
2  THA.1.3_1   THA  Thailand  THA.1_1  Amnat Charoen  จังหวัดอำนาจเจริญ   
3  THA.1.4_1   THA  Thailand  THA.1_1  Amnat Charoen  จังหวัดอำนาจเจริญ   
4  THA.1.5_1   THA  Thailand  THA.1_1  Amnat Charoen  จังหวัดอำนาจเจริญ   

                NAME_2 VARNAME_2             NL_NAME_2  TYPE_2 ENGTYPE_2  \
0             Chanuman        NA          อำเภอชานุมาน  Amphoe  District   
1           Hua Taphan        NA         อำเภอหัวตะพาน  Amphoe  District   
2             Lu Amnat        NA         อำเภอลืออำนาจ  Amphoe  District   
3  Muang Amnat Charoen        NA  อำเภอเมืองอำนาจเจริญ  Amphoe  District   
4     Pathum Ratwongsa        NA      อำเภอปทุมราชวงศา  Amphoe  District   

   CC_2    HASC_2                                           geometry  

In [68]:
# ทำ Spatial Join ระหว่างข้อมูลอำเภอกับข้อมูลจังหวัด
# เราจะใช้ gdf_districts_exercise3 เป็น 'left' layer (อำเภอ) และ gdf_exercise2 เป็น 'right' layer (จังหวัด)
# 'predicate="within"' ใช้สำหรับหาว่าอำเภอใดบ้างที่ 'อยู่ภายใน' ขอบเขตของจังหวัด
gdf_joined_exercise3 = gpd.sjoin(gdf_districts_exercise3, gdf_exercise2, how="inner", predicate="within")

print("ตัวอย่างข้อมูลที่เชื่อมโยงกัน (5 แถวแรก) แสดงอำเภอพร้อมข้อมูลจังหวัดที่อำเภอนั้นสังกัด:")
display(gdf_joined_exercise3.head())

ตัวอย่างข้อมูลที่เชื่อมโยงกัน (5 แถวแรก) แสดงอำเภอพร้อมข้อมูลจังหวัดที่อำเภอนั้นสังกัด:


,GID_2,GID_0,COUNTRY,GID_1,NAME_1,NL_NAME_1,NAME_2,VARNAME_2,NL_NAME_2,TYPE_2,...,ADM1ALT1EN,ADM1ALT2EN,ADM1ALT1TH,ADM1ALT2TH,ADM0_EN,ADM0_TH,ADM0_PCODE,date,validOn,validTo
15,THA.3.2_1,THA,Thailand,THA.3_1,Bangkok Metropolis,จังหวัดเชียงใหม่,Bang Kapi,NA,บางกะป,Khet,...,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT
16,THA.3.3_1,THA,Thailand,THA.3_1,Bangkok Metropolis,จังหวัดเชียงใหม่,Bang Khae,NA,บางแค,Khet,...,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT
17,THA.3.4_1,THA,Thailand,THA.3_1,Bangkok Metropolis,จังหวัดเชียงใหม่,Bang Khen,NA,บางเขน,Khet,...,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT
18,THA.3.5_1,THA,Thailand,THA.3_1,Bangkok Metropolis,จังหวัดเชียงใหม่,Bang Kho Laem,NA,บางคอแหลม,Khet,...,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT
21,THA.3.8_1,THA,Thailand,THA.3_1,Bangkok Metropolis,จังหวัดเชียงใหม่,Bang Rak,NA,บางรัก,Khet,...,None,None,None,None,Thailand,ประเทศไทย,TH,2019-02-18,2019-02-21,NaT


### แบบฝึกหัดที่ 4

In [83]:
import numpy as np
import pandas as pd

# ตรวจสอบให้แน่ใจว่า gdf_exercise2 มีอยู่จากการรันแบบฝึกหัดที่ 2
if 'gdf_exercise2' not in locals():
    print("กรุณารันเซลล์สำหรับ 'แบบฝึกหัดที่ 2' ก่อน เพื่อให้มีข้อมูล gdf_exercise2")
else:
    # สร้างค่า NDVI สมมุติสำหรับแต่ละจังหวัด
    # จำนวนค่า NDVI จะเท่ากับจำนวนแถวใน gdf_exercise2
    np.random.seed(42) # เพื่อให้ผลลัพธ์สามารถทำซ้ำได้
    gdf_exercise2['simulated_ndvi'] = np.random.uniform(0.1, 0.9, len(gdf_exercise2))

    print("ข้อมูลจังหวัดพร้อมค่า NDVI สมมุติ (5 แถวแรก):")
    display(gdf_exercise2[['ADM1_TH', 'simulated_ndvi']].head())

    # หาจังหวัดที่มีค่า NDVI สมมุติสูงสุด
    province_highest_ndvi = gdf_exercise2.loc[gdf_exercise2['simulated_ndvi'].idxmax()]

    print(f"\nจังหวัดที่มีค่า NDVI สมมุติสูงสุดคือ: {province_highest_ndvi['ADM1_TH']} ด้วยค่า {province_highest_ndvi['simulated_ndvi']:.2f}")

ข้อมูลจังหวัดพร้อมค่า NDVI สมมุติ (5 แถวแรก):


,ADM1_TH,simulated_ndvi
0,อำนาจเจริญ,0.399632
1,อ่างทอง,0.860571
2,กรุงเทพมหานคร,0.685595
3,บึงกาฬ,0.578927
4,บุรีรัมย์,0.224815



จังหวัดที่มีค่า NDVI สมมุติสูงสุดคือ: ตรัง ด้วยค่า 0.89
